# Finetuning LVH

Running `AS_detection/main.py` in this notebook.


## Setup

In [13]:
import os
import shutil
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torchvision
import tqdm
from sklearn import metrics

LVH_DIR = Path.cwd() / "LVH_detection"
sys.path.insert(0, str(LVH_DIR))

from dataset import EchoDatasetLVHDetection
from utils import evaluate, seed_worker, set_seed, train, validate

In [14]:
# Hyperparameters (defaults match the argparse options in main.py)
args = SimpleNamespace(
    data_dir="data",
    output_dir="output",
    clip_len=16,
    sampling_rate=1,
    num_clips=4,
    model_name="3dresnet18",
    rand_init=False,
    n_gpu=1,
    max_epochs=10,
    patience=1e4,
    lr=1e-4,
    batch_size=40,
    label_smoothing=0.0,
    use_class_weights=False,
    dropout_fc=False,
    augment=False,
    lpft="",
    ssl="ssl_pretraining/output/model-2026-1002",
    frac=1.0,
    weight_averaging=False,
    n_TTA=0,
    seed=0,
)
args

namespace(data_dir='data',
          output_dir='output',
          clip_len=16,
          sampling_rate=1,
          num_clips=4,
          model_name='3dresnet18',
          rand_init=False,
          n_gpu=1,
          max_epochs=10,
          patience=10000.0,
          lr=0.0001,
          batch_size=40,
          label_smoothing=0.0,
          use_class_weights=False,
          dropout_fc=False,
          augment=False,
          lpft='',
          ssl='ssl_pretraining/output/model-2026-1002',
          frac=1.0,
          weight_averaging=False,
          n_TTA=0,
          seed=0)

In [15]:
MODEL_NAME = args.model_name
MODEL_NAME += f"_frac-{args.frac}" if args.frac != 1.0 else ""
MODEL_NAME += "_pretr" if not args.rand_init else "_rand"
MODEL_NAME += f"_ssl-{args.ssl.split('/')[-1]}" if args.ssl != "" else ""
MODEL_NAME += "_aug" if args.augment else ""
MODEL_NAME += f"_clip-len-{args.clip_len}-stride-{args.sampling_rate}"
MODEL_NAME += f"_num-clips-{args.num_clips}"
MODEL_NAME += "_cw" if args.use_class_weights else ""
MODEL_NAME += f"_lr-{args.lr}"
MODEL_NAME += f"_{args.max_epochs}ep"
MODEL_NAME += f"_patience-{args.patience}" if args.patience != 1e4 else ""
MODEL_NAME += f"_bs-{args.batch_size}"
MODEL_NAME += f"_ls-{args.label_smoothing}" if args.label_smoothing != 0.0 else ""
MODEL_NAME += f"_drp-0.25" if args.dropout_fc else ""
MODEL_NAME += f"_weight-avg" if args.weight_averaging else ""
MODEL_NAME += f"_seed-{args.seed}" if args.seed != 0 else ""

MODEL_NAME += "_val-auc"

MODEL_NAME

'3dresnet18_pretr_ssl-model-2026-1002_clip-len-16-stride-1_num-clips-4_lr-0.0001_10ep_bs-40_val-auc'

In [16]:
# Create output directory for model (and delete if already exists)
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

model_dir = os.path.join(args.output_dir, MODEL_NAME)
if os.path.isdir(model_dir):
    shutil.rmtree(model_dir)
os.mkdir(model_dir)

model_dir

'output/3dresnet18_pretr_ssl-model-2026-1002_clip-len-16-stride-1_num-clips-4_lr-0.0001_10ep_bs-40_val-auc'

In [17]:
# Set all seeds for reproducibility
set_seed(args.seed)

# Set device (main.py hardcodes cuda:0; fall back so this also runs on a Mac)
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='mps')

## Create Model

In [18]:
# Create model
if args.model_name == "3dresnet18":
    weights = (
        torchvision.models.video.R3D_18_Weights.DEFAULT if not args.rand_init else None
    )
    model = torchvision.models.video.r3d_18(weights=weights)

    if args.dropout_fc:
        model.fc = torch.nn.Sequential(  # type: ignore
            torch.nn.Linear(512, 1), torch.nn.Dropout(0.25)
        )

        if args.lpft != "":
            # Linear probing fine-tuning (LPFT) weights are stored in a dict under the key "weights"
            weights = dict(torch.load(args.lpft, map_location="cpu")["weights"])

            model.fc[0].weight.data = weights["fc.0.weight"]
            model.fc[0].bias.data = weights["fc.0.bias"]

    else:
        model.fc = torch.nn.Linear(512, 1)
        if args.lpft != "":
            # Linear probing fine-tuning (LPFT) weights are stored in a dict under the key "weights"
            weights = dict(torch.load(args.lpft, map_location="cpu")["weights"])

            model.fc.weight.data = weights["fc.weight"]
            model.fc.bias.data = weights["fc.bias"]

    if args.ssl != "":
        checkpoints = [f for f in os.listdir(args.ssl) if f.endswith(".pt")]
        idx = np.argmax([int(f.split(".")[0].split("-")[1]) for f in checkpoints])
        weights = torch.load(
            os.path.join(args.ssl, checkpoints[idx]), map_location="cpu"
        )["weights"]
        weights = {
            k.replace("encoder.", ""): v for k, v in weights.items() if "encoder" in k
        }

        model.load_state_dict(weights, strict=False)


model = model.to(device)

In [19]:
model

VideoResNet(
  (stem): BasicStem(
    (0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (conv2): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (relu): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1

## Create Dataset

In [31]:
train_dataset = EchoDatasetLVHDetection(
    data_dir=args.data_dir,
    split="train",
    clip_len=args.clip_len,
    sampling_rate=args.sampling_rate,
    num_clips=args.num_clips,
    augment=args.augment,
    kinetics=(args.ssl == "") and (not args.rand_init),
)

False

In [22]:
args.data_dir

'data'

In [ ]:
import os
import random

import cv2
import numpy as np
import pandas as pd
import torch

from scipy.ndimage import rotate

from utils import load_video


class EchoDatasetLVHDetection(torch.utils.data.Dataset):
    """
    PyTorch dataset that loads echocardiogram videos for supervised finetuning for detecting LVH.

    Attributes
    ----------
    split: str
        Data split used to select studies that will be loaded (one of ["train", "val", "test"])
    clip_len : int
        Number of frames to form video clips for training (clip length)
    num_clips : int
        Number of clips to use for each video during inference (will be stacked along zeroth dimension with outputs averaged to form video-level prediction)
    sampling_rate : int
        Temporal "stride" when sampling frames to form clips (e.g., sampling_rate=1 samples consecutive frames)
    augment : bool
        Whether to perform data augmentation (spatially consistent augmentations to individual frames)
    kinetics : bool
        Whether or not to perform channel-wise normalization according to the Kinetics-400 training set mean and standard deviation
    video_dir : str
        Path to directory containing videos in .avi format
    label_df : pandas DataFrame
        Data frame containing file names and labels
    CLASSES : list[str]
        List of target class names
    mean : np.ndarray
        Channel-wise means of Kinetics-400 training set used for normalization
    std : np.ndarray
        Channel-wise standard deviations of Kinetics-400 training set used for normalization

    Methods
    -------
    _sample_frames(x)
        Subsample frames of video to form a video "clip" (returns multiple clips stacked along zeroth dimension during inference)
    _augment(x)
        Apply spatial augmentation to frames of video clip
    __len__
        Returns "length"/size of dataset
    __getitem__
        Returns dictionary {'x': [torch.FloatTensor] echo video clip,
                            'y': [torch.FloatTensor] classification label,
                            'acc_num': [str] accession number (study ID),
                            'video_num': [str] video ID to distinguish videos from same study,
                            'plax_prob': [float] predicted probability of video being from the PLAX view, as determined by view classifier}
    """

    def __init__(
        self,
        data_dir,
        split,
        clip_len=4,
        sampling_rate=1,
        num_clips=4,
        augment=False,
        kinetics=False,
    ):
        assert split in [
            "train",
            "val",
            "test",
        ], "Invalid split. Must be one of ['train', 'val', 'test']."

        self.split = split
        self.clip_len = clip_len
        self.num_clips = num_clips
        self.sampling_rate = sampling_rate
        self.augment = augment
        self.kinetics = kinetics

        self.video_dir = os.path.join(data_dir, "videos")
        self.label_df = pd.read_csv(os.path.join(data_dir, self.split + ".csv"))

    
        print(f"LVH class distribution ({self.split} split):")
        print(self.label_df["lvh"].value_counts(normalize=True))

        self.CLASSES = ["None", "LVH"]

    def _sample_frames(self, x):
        if self.clip_len is not None:
            if self.split == "train":
                if x.shape[0] > self.clip_len * self.sampling_rate:
                    start_idx = np.random.choice(
                        x.shape[0] - self.clip_len * self.sampling_rate, size=1
                    )[0]
                    x = x[
                        start_idx : (
                            start_idx + self.clip_len * self.sampling_rate
                        ) : self.sampling_rate
                    ]
                    x = np.transpose(x, (3, 0, 1, 2))
                else:
                    x = x[:: self.sampling_rate]
                    x = np.pad(
                        x,
                        ((0, self.clip_len - x.shape[0]), (0, 0), (0, 0), (0, 0)),
                        mode="constant",
                    )
                    x = np.transpose(x, (3, 0, 1, 2))
            else:
                if x.shape[0] >= self.clip_len * self.sampling_rate + self.num_clips:
                    start_indices = np.arange(
                        0,
                        x.shape[0] - self.clip_len * self.sampling_rate,
                        (x.shape[0] - self.clip_len * self.sampling_rate)
                        // self.num_clips,
                    )[: self.num_clips]
                    x = np.stack(
                        [
                            x[
                                start_idx : (
                                    start_idx + self.clip_len * self.sampling_rate
                                ) : self.sampling_rate
                            ]
                            for start_idx in start_indices
                        ],
                        axis=0,
                    )
                    x = np.transpose(x, (4, 0, 1, 2, 3))
                elif x.shape[0] > self.clip_len * self.sampling_rate:
                    x = x[:: self.sampling_rate]
                    x = x[: self.clip_len]
                    x = np.stack([x] * self.num_clips, axis=0)
                    x = np.transpose(x, (4, 0, 1, 2, 3))
                else:
                    x = x[:: self.sampling_rate]
                    x = np.pad(
                        x,
                        ((0, self.clip_len - x.shape[0]), (0, 0), (0, 0), (0, 0)),
                        mode="constant",
                    )
                    x = np.stack([x] * self.num_clips, axis=0)
                    x = np.transpose(x, (4, 0, 1, 2, 3))
        else:
            x = np.transpose(x, (3, 0, 1, 2))

        return x

    def _augment(self, x):
        # Zero-pad by up to 8 pixels
        pad = 8

        l, h, w, c = x.shape
        temp = np.zeros((l, h + 2 * pad, w + 2 * pad, c), dtype=x.dtype)
        temp[:, pad:-pad, pad:-pad, :] = x
        i, j = np.random.randint(0, 2 * pad, 2)
        x = temp[:, i : (i + h), j : (j + w), :]

        # Random horitzontal flip
        if random.uniform(0, 1) > 0.5:
            x = np.stack([cv2.flip(frame, 1) for frame in x], axis=0)

        # Random rotation between -10 and 10 degrees
        if random.uniform(0, 1) > 0.5:
            angle = np.random.choice(np.arange(-10, 11), size=1)[0]

            x = np.stack([rotate(frame, angle, reshape=False) for frame in x], axis=0)

        return x

    def __len__(self):
        return self.label_df.shape[0]

    def __getitem__(self, idx):
        # label_df columns for train/val/test:
        # HashedFileName, split, IVSd, LVIDd, LVPWd, max_wall, lvh
        fname, _, _, _, _, _, label = self.label_df.iloc[idx, :]
        acc_num = fname
        x = load_video(os.path.join(self.video_dir, fname))

        if self.augment:
            x = self._augment(x)

        x = self._sample_frames(x)

        x = (x - x.min()) / (x.max() - x.min())

        if self.kinetics:
            x -= self.mean.reshape(3, 1, 1, 1, 1)
            x /= self.std.reshape(3, 1, 1, 1, 1)

        y = np.array([label])

        return {
            "x": torch.from_numpy(x).float(),
            "y": torch.from_numpy(y).float(),
            "acc_num": acc_num,
        }


In [ ]:
train_dataset

In [ ]:
print(train_dataset.label_df.shape)
train_dataset.label_df.columns.tolist()

In [ ]:
print(train_dataset.label_df.head())
print(train_dataset.video_dir)
print(os.listdir(train_dataset.video_dir)[:5])

## Create DataLoader

In [ ]:
# Create loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=args.n_gpu * args.batch_size,
    shuffle=True,
    num_workers=8,
    worker_init_fn=seed_worker,
)

## Initialize Model Training Loop

In [ ]:
# Create csv documenting training history
history = pd.DataFrame(
    columns=[
        "epoch",
        "phase",
        "loss",
        "auroc",
        "aupr",
        "acc",
        "b_acc",
        "mcc",
        "precision",
        "recall",
        "f1",
    ]
)
history.to_csv(os.path.join(model_dir, "history.csv"), index=False)

In [ ]:
if args.use_class_weights:
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.sort(
            np.unique((train_dataset.label_df["av_stenosis"] == "Severe").values)
        ),
        y=(train_dataset.label_df["av_stenosis"] == "Severe").values,
    )

    print("Class weights:", class_weights)

    pos_weight = (class_weights / class_weights.min())[1]

    print("Normalized positive class weight:", pos_weight)

    loss_fxn = torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.Tensor([pos_weight]).to(device)
    )
    eval_loss_fxn = torch.nn.BCELoss(weight=torch.Tensor([pos_weight]).to(device))
else:
    loss_fxn = torch.nn.BCEWithLogitsLoss()
    eval_loss_fxn = torch.nn.BCELoss()

# Create optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

## Train Model

In [ ]:
def train(model, device, loss_fxn, optimizer, data_loader, history, epoch, model_dir):
    """Train model to classify LVH for one epoch.

    Parameters
    ----------
        model : PyTorch model
        device : PyTorch device
        loss_fxn : PyTorch loss function
        optimizer : PyTorch optimizer
        data_loader : PyTorch data loader
        history : pandas DataFrame
            Data frame containing history of training metrics
        epoch : int
            Current epoch number (1-K)
        model_dir : str
            Path to output directory where metrics, model weights, etc. will be stored
    Returns
    -------
        history : pandas DataFrame
            Updated history data frame with metrics from completed training epoch
    """
    model.train()

    pbar = tqdm.tqdm(
        enumerate(data_loader), total=len(data_loader), desc=f"Epoch {epoch}"
    )
    running_loss = 0.0
    y_true, y_hat = [], []
    acc_nums = []
    plax_probs = []
    for i, batch in pbar:
        x, y = batch["x"].to(device), batch["y"].to(device)
        acc_num = batch["acc_num"]

        # Forward pass
        out = model(x)

        # Compute loss
        loss = loss_fxn(out, y)

        # Backward pass
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        # Keep running sum of batch losses
        running_loss += loss.item()

        y_true.append(y.detach().cpu().numpy())
        y_hat.append(out.sigmoid().detach().cpu().numpy())
        acc_nums.append(acc_num)

        del out
        del loss

        pbar.set_postfix({"loss": running_loss / (i + 1)})  # , 'b_acc': balanced_acc})

    video_label_df = pd.DataFrame(
        {
            "y_true": np.concatenate(y_true).ravel(),
            "y_hat": np.concatenate(y_hat).ravel(),
            "acc_num": np.concatenate(acc_nums).ravel(),
        }
    )

    study_label_df = video_label_df.groupby(by=["acc_num"]).agg(
        {"y_true": np.mean, "y_hat": np.mean}
    )

    y_true = video_label_df["y_true"]
    y_hat = video_label_df["y_hat"]

    # AUROC
    auroc = metrics.roc_auc_score(y_true, y_hat)
    pr, re, _ = metrics.precision_recall_curve(y_true, y_hat)
    aupr = metrics.auc(re, pr)

    # Other metrics
    acc = metrics.accuracy_score(y_true, y_hat.round())
    b_acc = metrics.balanced_accuracy_score(y_true, y_hat.round())
    mcc = metrics.matthews_corrcoef(y_true, y_hat.round())
    pr, re, f1, _ = metrics.precision_recall_fscore_support(
        y_true, y_hat.round(), average="binary"
    )

    out_str = f"[VIDEO LEVEL]   AUROC: {auroc:.3f} | AUPR: {aupr:.3f} | Acc: {acc:.3f} | BAcc: {b_acc:.3f} | MCC: {mcc:.3f} | Precision: {pr:.3f} | Recall: {re:.3f} | F1: {f1:.3f}"
    print(out_str)

    ## REPEAT FOR STUDY LEVEL METRICS ##
    y_true = study_label_df["y_true"]
    y_hat = study_label_df["y_hat"]

    # AUROC
    auroc = metrics.roc_auc_score(y_true, y_hat)
    pr, re, _ = metrics.precision_recall_curve(y_true, y_hat)
    aupr = metrics.auc(re, pr)

    # Other metrics
    acc = metrics.accuracy_score(y_true, y_hat.round())
    b_acc = metrics.balanced_accuracy_score(y_true, y_hat.round())
    mcc = metrics.matthews_corrcoef(y_true, y_hat.round())
    pr, re, f1, _ = metrics.precision_recall_fscore_support(
        y_true, y_hat.round(), average="binary"
    )

    out_str = f"[STUDY LEVEL] AUROC: {auroc:.3f} | AUPR: {aupr:.3f} | Acc: {acc:.3f} | BAcc: {b_acc:.3f} | MCC: {mcc:.3f} | Precision: {pr:.3f} | Recall: {re:.3f} | F1: {f1:.3f}"
    print(out_str)

    # Append metrics to history df
    current_metrics = pd.DataFrame(
        [
            [
                epoch,
                "train",
                running_loss / (i + 1),
                auroc,
                aupr,
                acc,
                b_acc,
                mcc,
                pr,
                re,
                f1,
            ]
        ],
        columns=history.columns,
    )
    current_metrics.to_csv(
        os.path.join(model_dir, "history.csv"), mode="a", header=False, index=False
    )

    return history.append(current_metrics)

In [ ]:
data_loader = train_loader
epoch = 1

In [ ]:
model.train()

pbar = tqdm.tqdm(
    enumerate(data_loader), total=len(data_loader), desc=f"Epoch {epoch}"
)
running_loss = 0.0
y_true, y_hat = [], []
acc_nums = []
plax_probs = []

In [ ]:
# Train with early stopping
epoch = 1
early_stopping_dict = {"best_auroc": 0.0, "epochs_no_improve": 0}
best_model_wts = None
while (
    epoch <= args.max_epochs
    and early_stopping_dict["epochs_no_improve"] < args.patience
):
    history = train(
        model=model,
        device=device,
        loss_fxn=loss_fxn,
        optimizer=optimizer,
        data_loader=train_loader,
        history=history,
        epoch=epoch,
        model_dir=model_dir,
    )
    history, early_stopping_dict, best_model_wts = validate(
        model=model,
        device=device,
        loss_fxn=eval_loss_fxn,
        optimizer=optimizer,
        data_loader=val_loader,
        history=history,
        epoch=epoch,
        model_dir=model_dir,
        early_stopping_dict=early_stopping_dict,
        best_model_wts=best_model_wts,
    )

    epoch += 1

if args.weight_averaging:
    checkpoints = [
        torch.load(os.path.join(model_dir, f), map_location="cpu")["weights"]
        for f in os.listdir(model_dir)
        if f.endswith(".pt")
    ]
    for key in best_model_wts:
        best_model_wts[key] = torch.stack(
            [chkpt[key] for chkpt in checkpoints], dim=0
        ).sum(dim=0) / len(checkpoints)


## Validate Model

In [ ]:
def validate(model, device, loss_fxn, optimizer, data_loader, history, epoch, model_dir, early_stopping_dict, best_model_wts):
    """Evaluate LVH model on validation set, saving weights if val loss has improved.

    Parameters
    ----------
        model : PyTorch model
        device : PyTorch device
        loss_fxn : PyTorch loss function
        optimizer : PyTorch optimizer
        data_loader : PyTorch data loader
        history : pandas DataFrame
            Data frame containing history of training metrics
        epoch : int
            Current epoch number (1-K)
        model_dir : str
            Path to output directory where metrics, model weights, etc. will be stored
        early_stopping_dict: dict
            Dictionary with keys "best_loss" and "epochs_no_improve" for early stopping and model checkpointing
        best_model_wts: dict
            Dictionary containing PyTorch model weights as returned by `model.state_dict()`
    Returns
    -------
        history : pandas DataFrame
            Updated history data frame with metrics from completed training epoch
        early_stopping_dict: dict
            Updated dictionary with information for early stopping and model checkpointing
        best_model_wts: dict
            Updated dictionary containing best PyTorch model weights
    """
    model.eval()

    pbar = tqdm.tqdm(enumerate(data_loader), total=len(data_loader), desc=f'[VAL] Epoch {epoch}')
    running_loss = 0.
    y_true, y_hat = [], []
    acc_nums = []
    plax_probs = []
    with torch.no_grad():
        for i, batch in pbar:
            x, y = batch['x'].to(device), batch['y'].to(device)
            acc_num = batch['acc_num']

            # Forward pass
            out = torch.stack([model(x[:, :, clip, :, :, :]) for clip in range(x.shape[2])], dim=0)
            
            # Get softmax probabilities and video-wise average over clip dimension
            out = out.sigmoid().mean(dim=0)

            # Compute loss
            loss = loss_fxn(out, y)

            # Keep running sum of batch losses
            running_loss += loss.item()

            y_true.append(y.detach().cpu().numpy())
            y_hat.append(out.detach().cpu().numpy())
            acc_nums.append(acc_num)

            del out
            del loss

            pbar.set_postfix({'loss': running_loss / (i + 1)}) #, 'b_acc': balanced_acc})

    video_label_df = pd.DataFrame({'y_true': np.concatenate(y_true).ravel(), 'y_hat': np.concatenate(y_hat).ravel(), 'acc_num': np.concatenate(acc_nums).ravel()})

    study_label_df = video_label_df.groupby(by=['acc_num']).agg({'y_true': np.mean, 'y_hat': np.mean})

    y_true = video_label_df['y_true']
    y_hat = video_label_df['y_hat']

    # AUROC
    auroc = metrics.roc_auc_score(y_true, y_hat)
    pr, re, _ = metrics.precision_recall_curve(y_true, y_hat)
    aupr = metrics.auc(re, pr)

    # Other metrics
    acc = metrics.accuracy_score(y_true, y_hat.round())
    b_acc = metrics.balanced_accuracy_score(y_true, y_hat.round())
    mcc = metrics.matthews_corrcoef(y_true, y_hat.round())
    pr, re, f1, _ = metrics.precision_recall_fscore_support(y_true, y_hat.round(), average='binary')

    out_str = f'[VIDEO LEVEL]   AUROC: {auroc:.3f} | AUPR: {aupr:.3f} | Acc: {acc:.3f} | BAcc: {b_acc:.3f} | MCC: {mcc:.3f} | Precision: {pr:.3f} | Recall: {re:.3f} | F1: {f1:.3f}'
    print(out_str)

    ## REPEAT FOR STUDY LEVEL METRICS ##
    y_true = study_label_df['y_true']
    y_hat = study_label_df['y_hat']

    # AUROC
    auroc = metrics.roc_auc_score(y_true, y_hat)
    pr, re, _ = metrics.precision_recall_curve(y_true, y_hat)
    aupr = metrics.auc(re, pr)

    # Other metrics
    acc = metrics.accuracy_score(y_true, y_hat.round())
    b_acc = metrics.balanced_accuracy_score(y_true, y_hat.round())
    mcc = metrics.matthews_corrcoef(y_true, y_hat.round())
    pr, re, f1, _ = metrics.precision_recall_fscore_support(y_true, y_hat.round(), average='binary')

    out_str = f'[STUDY LEVEL] AUROC: {auroc:.3f} | AUPR: {aupr:.3f} | Acc: {acc:.3f} | BAcc: {b_acc:.3f} | MCC: {mcc:.3f} | Precision: {pr:.3f} | Recall: {re:.3f} | F1: {f1:.3f}'
    print(out_str)

    # Append metrics to history df
    val_loss = running_loss / (i+1)

    current_metrics = pd.DataFrame([[epoch, 'val', running_loss/(i+1), auroc, aupr, acc, b_acc, mcc, pr, re, f1]], columns=history.columns)
    current_metrics.to_csv(os.path.join(model_dir, 'history.csv'), mode='a', header=False, index=False)

    # Early stopping: save model weights only when val AUC has improved
    if auroc > early_stopping_dict['best_auroc']:
        print(f'EARLY STOPPING: AUROC has improved from {round(early_stopping_dict["best_auroc"], 3)} to {round(auroc, 3)}! Saving weights.')
        early_stopping_dict['epochs_no_improve'] = 0
        early_stopping_dict['best_auroc'] = auroc
        best_model_wts = deepcopy(model.state_dict()) if not isinstance(model, torch.nn.DataParallel) else deepcopy(model.module.state_dict())
        torch.save({'weights': best_model_wts, 'optimizer': optimizer.state_dict()}, os.path.join(model_dir, f'chkpt_epoch-{epoch}.pt'))
    else:
        print(f'EARLY STOPPING: AUROC has not improved from {round(early_stopping_dict["best_auroc"], 3)}')
        early_stopping_dict['epochs_no_improve'] += 1

    return history.append(current_metrics), early_stopping_dict, best_model_wts
